In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import os

# ============================================
# KONFIGURATION
# ============================================
TARGET_NHDA_ID = "09186_5"

COMBINED_GPKG   = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI.gpkg"
CONSTRUCTION_GPKG = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\New_Housing_Development_Areas\NHDA_with_construction_years_RF_AUC.gpkg"

OUTPUT_DIR = r"C:\Users\agz90fk\Documents\EO4CAM\07_Abbildungen\Masterarbeit\Env_Con_Example"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Sonderwerte für construction_start_year, analog zum bestehenden Skript
CONSTRUCTION_SPECIAL_MAP = {
    'AUC_2015': 2014,
    'AUC_2016': 2015,
}


# ============================================
# HILFSFUNKTIONEN
# ============================================
def find_year_columns(frame, prefix):
    """Findet alle Spalten der Form <prefix>_<jahr> (z.B. LST_2019) und gibt {jahr: spaltenname} zurück."""
    pattern = re.compile(rf"^{re.escape(prefix)}_(\d{{4}})$")
    result = {}
    for col in frame.columns:
        m = pattern.match(col)
        if m:
            result[int(m.group(1))] = col
    return result


def find_uncertainty_columns(frame, prefix):
    """
    Sucht optionale Unsicherheits-/Std-Spalten zu einem Jahres-Prefix, z.B.
    LST_2019_std, LST_std_2019, LST_2019_sd -> {jahr: spaltenname}
    Gibt ein leeres dict zurück, wenn nichts gefunden wird (dann wird kein Band geplottet).
    """
    patterns = [
        rf"^{re.escape(prefix)}_(\d{{4}})_std$",
        rf"^{re.escape(prefix)}_std_(\d{{4}})$",
        rf"^{re.escape(prefix)}_(\d{{4}})_sd$",
        rf"^{re.escape(prefix)}_(\d{{4}})_stdev$",
    ]
    result = {}
    for pat in patterns:
        regex = re.compile(pat)
        for col in frame.columns:
            m = regex.match(col)
            if m:
                result[int(m.group(1))] = col
    return result


def extract_series(row, year_cols, std_cols=None):
    """Baut aus einer einzelnen Zeile (Series) eine long-Tabelle jahr -> wert (+ std, falls vorhanden)."""
    records = []
    for year, col in sorted(year_cols.items()):
        value = row.get(col, np.nan)
        std_val = np.nan
        if std_cols and year in std_cols:
            std_val = row.get(std_cols[year], np.nan)
        records.append({'year': year, 'value': value, 'std': std_val})
    return pd.DataFrame(records)


# ============================================
# 1. DATEN LADEN
# ============================================
print("=" * 80)
print(f"EINZEL-NHDA VERLAUF: {TARGET_NHDA_ID}")
print("=" * 80)

for path in [COMBINED_GPKG, CONSTRUCTION_GPKG]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Datei nicht gefunden: {path}")

gdf = gpd.read_file(COMBINED_GPKG)
gdf['nhda_id'] = gdf['nhda_id'].astype(str)

gdf_const = gpd.read_file(CONSTRUCTION_GPKG)

# ID-Spalte im Construction-GPKG identifizieren (analog zum Hauptskript)
id_candidates = ['nhda_id', 'cluster_id', 'nda_id']
id_col = next((c for c in id_candidates if c in gdf_const.columns), None)
if id_col is None:
    raise ValueError(f"Keine ID-Spalte im Construction-GPKG gefunden. Versucht: {id_candidates}")

if 'construction_start_year' not in gdf_const.columns:
    raise ValueError("Spalte 'construction_start_year' nicht im Construction-GPKG gefunden.")

gdf_const['nhda_id'] = gdf_const[id_col].astype(str)

const_row = gdf_const[gdf_const['nhda_id'] == TARGET_NHDA_ID]
if const_row.empty:
    raise ValueError(f"NHDA-ID {TARGET_NHDA_ID} nicht im Construction-GPKG gefunden.")

raw_construction = str(const_row.iloc[0]['construction_start_year']).strip()
construction_year = CONSTRUCTION_SPECIAL_MAP.get(raw_construction, raw_construction)
try:
    construction_year = float(construction_year)
except (TypeError, ValueError):
    raise ValueError(f"construction_start_year '{raw_construction}' konnte nicht in ein Jahr umgewandelt werden.")

print(f"   Construction start (roh):      {raw_construction}")
print(f"   Construction start (verwendet): {construction_year}")

# ============================================
# 2. ZEILEN FÜR DIE GEWÄHLTE NHDA FILTERN (NHDA + RA)
# ============================================
subset = gdf[gdf['nhda_id'] == TARGET_NHDA_ID].copy()
if subset.empty:
    raise ValueError(f"NHDA-ID {TARGET_NHDA_ID} nicht im Comparison-GPKG gefunden.")

# Typ-Spalte identifizieren (NHDA vs. RA)
type_col = None
for candidate in ['type', 'area_type']:
    if candidate in subset.columns:
        type_col = candidate
        break
if type_col is None:
    raise ValueError("Keine Typ-Spalte ('type' oder 'area_type') gefunden, um NHDA/RA zu unterscheiden.")

print(f"   Gefundene Zeilen für {TARGET_NHDA_ID}: {len(subset)}")
print(f"   Typen: {subset[type_col].unique().tolist()}")

# Optionaler Ortsname für den Plot-Titel
place_col = next((c for c in ['gemeinde', 'kommune', 'ort', 'place_name', 'location'] if c in subset.columns), None)
place_name = subset.iloc[0][place_col] if place_col else None

# ============================================
# 3. SPALTEN FÜR LST / NDVI / DIFFERENCE FINDEN
# ============================================
lst_year_cols   = find_year_columns(gdf, 'LST')
ndvi_year_cols  = find_year_columns(gdf, 'NDVI')
lst_diff_cols   = find_year_columns(gdf, 'difference_LST')
ndvi_diff_cols  = find_year_columns(gdf, 'difference_NDVI')

lst_std_cols  = find_uncertainty_columns(gdf, 'LST')
ndvi_std_cols = find_uncertainty_columns(gdf, 'NDVI')
lst_diff_std_cols  = find_uncertainty_columns(gdf, 'difference_LST')
ndvi_diff_std_cols = find_uncertainty_columns(gdf, 'difference_NDVI')

print(f"   LST-Jahresspalten gefunden:  {len(lst_year_cols)}")
print(f"   NDVI-Jahresspalten gefunden: {len(ndvi_year_cols)}")
print(f"   difference_LST gefunden:    {len(lst_diff_cols)}")
print(f"   difference_NDVI gefunden:   {len(ndvi_diff_cols)}")
if not lst_std_cols and not ndvi_std_cols:
    print("   Hinweis: Keine Std-/Unsicherheitsspalten gefunden -> Bänder werden ggf. leer bleiben.")

# ============================================
# 4. LONG-TABELLEN BAUEN (getrennt für NHDA und RA)
# ============================================
def get_row(area_type):
    rows = subset[subset[type_col].astype(str) == area_type]
    if rows.empty:
        return None
    return rows.iloc[0]

row_nhda = get_row('NHDA')
row_ra = get_row('RA')

lst_nhda = extract_series(row_nhda, lst_year_cols, lst_std_cols) if row_nhda is not None else pd.DataFrame()
lst_ra   = extract_series(row_ra,   lst_year_cols, lst_std_cols) if row_ra is not None else pd.DataFrame()
ndvi_nhda = extract_series(row_nhda, ndvi_year_cols, ndvi_std_cols) if row_nhda is not None else pd.DataFrame()
ndvi_ra   = extract_series(row_ra,   ndvi_year_cols, ndvi_std_cols) if row_ra is not None else pd.DataFrame()

# Differenzen liegen typischerweise nur in der NHDA-Zeile (bereits NHDA - RA)
diff_row = row_nhda if row_nhda is not None else row_ra
lst_diff  = extract_series(diff_row, lst_diff_cols, lst_diff_std_cols) if diff_row is not None else pd.DataFrame()
ndvi_diff = extract_series(diff_row, ndvi_diff_cols, ndvi_diff_std_cols) if diff_row is not None else pd.DataFrame()

for name, frame in [('LST NHDA', lst_nhda), ('LST RA', lst_ra),
                     ('NDVI NHDA', ndvi_nhda), ('NDVI RA', ndvi_ra),
                     ('diff LST', lst_diff), ('diff NDVI', ndvi_diff)]:
    frame.dropna(subset=['value'], inplace=True)

# ============================================
# 5. PLOT
# ============================================
title_place = f" in {place_name}" if place_name else ""
fig, axes = plt.subplots(4, 1, figsize=(9, 16), sharex=False)
fig.suptitle(f"New Housing Development Area ({TARGET_NHDA_ID}){title_place}", fontweight='bold', fontsize=13)

def plot_two_lines(ax, series_a, series_b, label_a, label_b, color_a, color_b, ylabel, title):
    for series, label, color in [(series_a, label_a, color_a), (series_b, label_b, color_b)]:
        if series.empty:
            continue
        ax.plot(series['year'], series['value'], marker='o', color=color, label=label, linewidth=2)
        if series['std'].notna().any():
            ax.fill_between(
                series['year'],
                series['value'] - series['std'],
                series['value'] + series['std'],
                color=color, alpha=0.2
            )
    ax.axvline(construction_year, linestyle='--', color='olive', linewidth=1.8, label='Construction start')
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Year')
    ax.set_title(title, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend()

def plot_diff(ax, series, color, ylabel, title, diff_label):
    if not series.empty:
        ax.plot(series['year'], series['value'], marker='o', color=color, linewidth=2, label=diff_label)
        if series['std'].notna().any():
            ax.fill_between(
                series['year'],
                series['value'] - series['std'],
                series['value'] + series['std'],
                color=color, alpha=0.2
            )
    ax.axhline(0, linestyle='--', color='red', linewidth=1.5)
    ax.axvline(construction_year, linestyle='--', color='olive', linewidth=1.8, label='Construction start')
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Year')
    ax.set_title(title, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend()

plot_two_lines(
    axes[0], lst_nhda, lst_ra, 'NHDA', 'RA', 'red', 'blue',
    'LST [°C]', 'LST'
)
plot_diff(
    axes[1], lst_diff, 'darkred', 'ΔLST [°C]', 'ΔLST', 'Difference (LST)'
)
plot_two_lines(
    axes[2], ndvi_nhda, ndvi_ra, 'NHDA', 'RA', 'red', 'blue',
    'NDVI', 'NDVI'
)
plot_diff(
    axes[3], ndvi_diff, 'darkgreen', 'ΔNDVI', 'ΔNDVI', 'Difference (NDVI)'
)

plt.tight_layout(rect=[0, 0, 1, 0.97])
out_path = f"{OUTPUT_DIR}/{TARGET_NHDA_ID}_trajectory.png"
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"\n   Plot gespeichert: {out_path}")
print("\nFERTIG")

EINZEL-NHDA VERLAUF: 09186_5
   Construction start (roh):      2020
   Construction start (verwendet): 2020.0
   Gefundene Zeilen für 09186_5: 2
   Typen: ['NHDA', 'RA']
   LST-Jahresspalten gefunden:  0
   NDVI-Jahresspalten gefunden: 0
   difference_LST gefunden:    11
   difference_NDVI gefunden:   10
   Hinweis: Keine Std-/Unsicherheitsspalten gefunden -> Bänder werden ggf. leer bleiben.


KeyError: ['value']